In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("human_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("human_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate key miRNA count for each lncRNA
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'stomach']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/human/BC_top40pct_human_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract key miRNA counts
    ess_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(ess_valid), 'key_miRNA'
    ]
    bg_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(bg_valid), 'key_miRNA'
    ]

    # Perform one-sided Mann-Whitney U test:
    # alternative='greater' means testing whether essential lncRNAs
    # interact with more key miRNAs than background lncRNAs
    u_stat, p_value = mannwhitneyu(
        ess_key_counts,
        bg_key_counts,
        alternative='greater'
    )

    # Save summary statistics
    results.append({
        'tissue': t,
        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),
        'mean_key_ess': ess_key_counts.mean(),
        'median_key_ess': ess_key_counts.median(),
        'mean_key_bg': bg_key_counts.mean(),
        'median_key_bg': bg_key_counts.median(),
        'u_stat': u_stat,
        'p_value': p_value
    })

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("human_ess_vs_background_key_miRNA_mwu.csv", index=False)

print(results_df)


In [1]:
import pandas as pd
from scipy.stats import mannwhitneyu

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("mouse_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("mouse_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate key miRNA count for each lncRNA
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'brain']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/mouse/BC_top60pct_mouse_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract key miRNA counts
    ess_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(ess_valid), 'key_miRNA'
    ]
    bg_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(bg_valid), 'key_miRNA'
    ]

    # Perform one-sided Mann-Whitney U test:
    # alternative='greater' means testing whether essential lncRNAs
    # interact with more key miRNAs than background lncRNAs
    u_stat, p_value = mannwhitneyu(
        ess_key_counts,
        bg_key_counts,
        alternative='greater'
    )

    # Save summary statistics
    results.append({
        'tissue': t,
        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),
        'mean_key_ess': ess_key_counts.mean(),
        'median_key_ess': ess_key_counts.median(),
        'mean_key_bg': bg_key_counts.mean(),
        'median_key_bg': bg_key_counts.median(),
        'u_stat': u_stat,
        'p_value': p_value
    })

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("mouse_ess_vs_background_key_miRNA_mwu.csv", index=False)

print(results_df)


  tissue  n_ess_total  n_ess_used  n_bg_total  n_bg_used  mean_key_ess  \
0  heart         4154         254       24871        899      2.814961   
1   lung         6202         414       22823        739      2.927536   
2  brain         4243         324       24782        829      3.098765   

   median_key_ess  mean_key_bg  median_key_bg    u_stat   p_value  
0             1.0      2.27030            1.0  115997.0  0.343713  
1             1.0      2.08931            1.0  162265.5  0.038259  
2             1.0      2.11339            1.0  139716.5  0.135175  
